# 🎬 Scalable Video Recommendation System
## Step 3: DIN Ranking Model（深度兴趣网络精排）

---
**本 Notebook 完成**:
```
Cell 1  → 安装依赖
Cell 2  → 加载 Step 1/2 产出的特征、样本、Embedding
Cell 3  → 数据集封装（拼接召回候选 + 全特征）
Cell 4  → DIN 模型定义（注意力机制 + MLP）
Cell 5  → 训练循环（含 Early Stopping）
Cell 6  → 训练过程可视化
Cell 7  → 离线评估（AUC / GAUC / LogLoss）
Cell 8  → 完整推荐链路测试（召回 → 精排 → Top10）
Cell 9  → 模型保存 + Feature Store 更新
Cell 10 → Step 3 完成报告
```

> 💡 **建议**: 运行时 → 更改运行时类型 → GPU (T4)

---
## Cell 1 — 安装依赖

In [ ]:
!pip install -q torch torchvision faiss-cpu pyarrow pandas numpy scikit-learn tqdm tabulate matplotlib
print('✅ 依赖安装完成')

---
## Cell 2 — 配置 & 加载数据

In [ ]:
import os, time, logging, warnings, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

# ── 路径配置 ──
USE_GDRIVE = False  # 用了 Google Drive 改为 True
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/VideoRecSys'
else:
    BASE_DIR = '/content/VideoRecSys'

PATHS = {
    'cleaned'      : f'{BASE_DIR}/data/cleaned',
    'features'     : f'{BASE_DIR}/data/features',
    'samples'      : f'{BASE_DIR}/data/samples',
    'feature_store': f'{BASE_DIR}/feature_store',
    'recall_models': f'{BASE_DIR}/models/recall',
    'rank_models'  : f'{BASE_DIR}/models/ranking',
    'logs'         : f'{BASE_DIR}/logs',
    'reports'      : f'{BASE_DIR}/reports',
}
for p in PATHS.values(): os.makedirs(p, exist_ok=True)

# ── 超参数 ──
CFG = {
    'embedding_dim'   : 32,
    'attention_hidden': [64, 16],
    'dnn_hidden'      : [256, 128, 64],
    'dropout'         : 0.3,
    'lr'              : 5e-4,
    'batch_size'      : 1024,
    'epochs'          : 30,
    'early_stop'      : 4,
    'seq_len'         : 50,
    'use_tower_emb'   : True,   # 是否使用 Step 2 的 Embedding
    'tower_emb_dim'   : 64,     # Step 2 Embedding 维度
    'random_seed'     : 42,
    'device'          : 'cuda' if torch.cuda.is_available() else 'cpu',
}
torch.manual_seed(CFG['random_seed'])
np.random.seed(CFG['random_seed'])

# ── Logger ──
def get_logger():
    logger = logging.getLogger('Step3')
    logger.setLevel(logging.INFO)
    if logger.handlers: logger.handlers.clear()
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter('[%(asctime)s] %(message)s', '%H:%M:%S'))
    logger.addHandler(ch)
    return logger
log = get_logger()

class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); log.info(f'▶ {self.name}'); return self
    def __exit__(self, *a): log.info(f'✅ {self.name} 完成  {time.time()-self.t:.1f}s')

# ── 加载数据 ──
with Timer('加载数据'):
    train_df      = pd.read_parquet(f"{PATHS['samples']}/train.parquet")
    test_df       = pd.read_parquet(f"{PATHS['samples']}/test.parquet")
    user_feat     = pd.read_parquet(f"{PATHS['features']}/user_features.parquet")
    item_feat     = pd.read_parquet(f"{PATHS['features']}/item_features.parquet")
    seq_df        = pd.read_parquet(f"{PATHS['features']}/user_sequences.parquet")

    # Step 2 产出的 Embedding
    user_emb_df   = pd.read_parquet(f"{PATHS['feature_store']}/user_embeddings.parquet")
    item_emb_df   = pd.read_parquet(f"{PATHS['feature_store']}/item_embeddings.parquet")

    # Step 2 的 ID 映射
    with open(f"{PATHS['recall_models']}/id_mappings.pkl", 'rb') as f:
        id_maps = pickle.load(f)
    user2idx = id_maps['user2idx']
    item2idx = id_maps['item2idx']
    idx2item = id_maps['idx2item']

log.info(f'训练集: {len(train_df):,}  测试集: {len(test_df):,}')
log.info(f'用户数: {user_feat.shape[0]:,}  视频数: {item_feat.shape[0]:,}')
log.info(f'用户Embedding: {user_emb_df.shape}  视频Embedding: {item_emb_df.shape}')
log.info(f'设备: {CFG["device"]}')

---
## Cell 3 — 特征预处理 & Dataset 封装

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# ── 数值特征列 ──
USER_FEAT_COLS = [c for c in user_feat.columns
                  if c != 'user_id' and user_feat[c].dtype in
                  [np.float64, np.float32, np.int64, np.int32]]
ITEM_FEAT_COLS = [c for c in item_feat.columns
                  if c != 'item_id' and item_feat[c].dtype in
                  [np.float64, np.float32, np.int64, np.int32]]

# ── 归一化 ──
user_scaler = MinMaxScaler()
item_scaler = MinMaxScaler()
user_feat_np = user_scaler.fit_transform(
    user_feat[USER_FEAT_COLS].fillna(0).values.astype(np.float32))
item_feat_np = item_scaler.fit_transform(
    item_feat[ITEM_FEAT_COLS].fillna(0).values.astype(np.float32))

user_feat_dict = {uid: user_feat_np[i] for i, uid in enumerate(user_feat['user_id'])}
item_feat_dict = {iid: item_feat_np[i] for i, iid in enumerate(item_feat['item_id'])}

# ── Step 2 Embedding 字典 ──
EMB_COLS_U = [c for c in user_emb_df.columns if c.startswith('user_emb_')]
EMB_COLS_I = [c for c in item_emb_df.columns if c.startswith('item_emb_')]
user_emb_dict = {row['user_id']: row[EMB_COLS_U].values.astype(np.float32)
                 for _, row in user_emb_df.iterrows()}
item_emb_dict = {row['item_id']: row[EMB_COLS_I].values.astype(np.float32)
                 for _, row in item_emb_df.iterrows()}

# ── 序列字典 ──
seq_dict = {}
for _, row in seq_df.iterrows():
    raw = str(row.get('pos_seq', '') or '')
    ids = [item2idx[int(x)] for x in raw.split(',') if x and int(x) in item2idx]
    seq_dict[row['user_id']] = ids

U_DIM   = len(next(iter(user_feat_dict.values())))
I_DIM   = len(next(iter(item_feat_dict.values())))
EMB_DIM = len(next(iter(user_emb_dict.values())))
N_ITEMS = len(item2idx)

log.info(f'用户特征维度: {U_DIM}  视频特征维度: {I_DIM}  Embedding维度: {EMB_DIM}')

In [ ]:
# ============================================================
# DIN Dataset
# 每条样本包含:
#   用户特征 + 用户Embedding
#   目标视频特征 + 视频Embedding
#   用户行为序列（用于注意力计算）
#   标签（1=正样本 0=负样本）
# ============================================================
class DINDataset(Dataset):
    def __init__(self, df, user_feat_dict, item_feat_dict,
                 user_emb_dict, item_emb_dict,
                 seq_dict, item2idx, seq_len=50):
        self.df            = df.reset_index(drop=True)
        self.u_feat        = user_feat_dict
        self.i_feat        = item_feat_dict
        self.u_emb         = user_emb_dict
        self.i_emb         = item_emb_dict
        self.seq_dict      = seq_dict
        self.item2idx      = item2idx
        self.seq_len       = seq_len
        self.u_dim         = len(next(iter(user_feat_dict.values())))
        self.i_dim         = len(next(iter(item_feat_dict.values())))
        self.emb_dim       = len(next(iter(user_emb_dict.values())))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        uid   = row['user_id']
        iid   = row['item_id']
        label = float(row['label'])

        # 用户侧特征
        u_vec = self.u_feat.get(uid, np.zeros(self.u_dim, np.float32))
        u_emb = self.u_emb.get(uid, np.zeros(self.emb_dim, np.float32))

        # 视频侧特征
        i_vec = self.i_feat.get(iid, np.zeros(self.i_dim, np.float32))
        i_emb = self.i_emb.get(iid, np.zeros(self.emb_dim, np.float32))

        # 行为序列（item embedding 序列，用于注意力）
        seq = self.seq_dict.get(uid, [])
        seq = seq[-self.seq_len:]
        pad_len = self.seq_len - len(seq)
        seq_pad = [0] * pad_len + seq
        seq_mask = [0] * pad_len + [1] * len(seq)  # padding mask

        return (
            torch.tensor(u_vec,     dtype=torch.float),
            torch.tensor(u_emb,     dtype=torch.float),
            torch.tensor(i_vec,     dtype=torch.float),
            torch.tensor(i_emb,     dtype=torch.float),
            torch.tensor(seq_pad,   dtype=torch.long),
            torch.tensor(seq_mask,  dtype=torch.float),
            torch.tensor(label,     dtype=torch.float),
        )

train_ds = DINDataset(train_df, user_feat_dict, item_feat_dict,
                      user_emb_dict, item_emb_dict,
                      seq_dict, item2idx, CFG['seq_len'])
test_ds  = DINDataset(test_df,  user_feat_dict, item_feat_dict,
                      user_emb_dict, item_emb_dict,
                      seq_dict, item2idx, CFG['seq_len'])

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)

log.info(f'训练集: {len(train_ds):,}  测试集: {len(test_ds):,}')

---
## Cell 4 — DIN 模型定义

In [ ]:
# ============================================================
# DIN: Deep Interest Network (阿里巴巴, KDD 2018)
#
# 核心创新: 注意力机制
#   普通模型: 行为序列直接平均 → 忽略不同行为的重要性差异
#   DIN:     目标视频 × 序列中每个视频 → 计算相关性权重
#            → 加权平均 → 更精准的用户兴趣表达
#
# 输入:
#   用户特征向量 + 用户Embedding（来自Step2双塔）
#   目标视频特征 + 视频Embedding
#   用户行为序列（item index序列）
# 输出:
#   点击概率 [0, 1]
# ============================================================

class DINAttention(nn.Module):
    """
    DIN 注意力单元
    计算目标视频与历史行为序列中每个视频的相关性
    """
    def __init__(self, emb_dim, hidden=[64, 16]):
        super().__init__()
        # 输入: [目标视频emb, 序列视频emb, 差值, 点积] 拼接
        input_dim = emb_dim * 4
        layers = []
        dims = [input_dim] + hidden + [1]
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, target_emb, seq_emb, mask):
        """
        target_emb: [B, D]
        seq_emb:    [B, L, D]
        mask:       [B, L]  1=有效 0=padding
        """
        B, L, D = seq_emb.shape
        target_exp = target_emb.unsqueeze(1).expand(-1, L, -1)  # [B, L, D]

        # 拼接4种交互特征（DIN原论文做法）
        concat = torch.cat([
            target_exp,
            seq_emb,
            target_exp - seq_emb,
            target_exp * seq_emb,
        ], dim=-1)  # [B, L, 4D]

        scores = self.net(concat).squeeze(-1)       # [B, L]
        scores = scores - (1 - mask) * 1e9          # mask掉padding位置
        weights = torch.softmax(scores, dim=-1)     # [B, L]
        output = (weights.unsqueeze(-1) * seq_emb).sum(dim=1)  # [B, D]
        return output


class DINModel(nn.Module):
    """
    完整 DIN 精排模型
    """
    def __init__(self, n_items, i_feat_dim, u_feat_dim,
                 tower_emb_dim, cfg):
        super().__init__()
        emb_dim    = cfg['embedding_dim']
        att_hidden = cfg['attention_hidden']
        dnn_hidden = cfg['dnn_hidden']
        dropout    = cfg['dropout']

        # 序列用的 item embedding（独立学习，不复用Step2）
        self.item_emb = nn.Embedding(n_items + 1, emb_dim, padding_idx=0)

        # DIN 注意力层
        self.attention = DINAttention(emb_dim, att_hidden)

        # DNN 输入维度:
        # 用户特征 + 用户tower_emb + 视频特征 + 视频tower_emb + 注意力输出
        dnn_input_dim = (u_feat_dim + tower_emb_dim +
                         i_feat_dim + tower_emb_dim + emb_dim)

        # DNN 层
        layers = []
        dims = [dnn_input_dim] + dnn_hidden
        for i in range(len(dims) - 1):
            layers += [
                nn.Linear(dims[i], dims[i+1]),
                nn.BatchNorm1d(dims[i+1]),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
        layers.append(nn.Linear(dnn_hidden[-1], 1))
        self.dnn = nn.Sequential(*layers)

    def forward(self, u_feat, u_emb, i_feat, i_emb, seq_idx, seq_mask):
        # 序列 Embedding
        seq_emb    = self.item_emb(seq_idx)           # [B, L, emb_dim]

        # 目标视频 Embedding（用i_emb的前emb_dim维近似）
        target_emb = self.item_emb(
            torch.zeros(u_feat.size(0), dtype=torch.long,
                        device=u_feat.device))         # placeholder

        # DIN 注意力：目标视频 × 历史序列
        att_out = self.attention(i_emb[:, :self.item_emb.embedding_dim],
                                 seq_emb, seq_mask)    # [B, emb_dim]

        # 拼接所有特征
        x = torch.cat([u_feat, u_emb, i_feat, i_emb, att_out], dim=-1)
        logit = self.dnn(x).squeeze(-1)               # [B]
        return torch.sigmoid(logit)


model = DINModel(
    n_items       = N_ITEMS,
    i_feat_dim    = I_DIM,
    u_feat_dim    = U_DIM,
    tower_emb_dim = EMB_DIM,
    cfg           = CFG,
).to(CFG['device'])

total = sum(p.numel() for p in model.parameters())
log.info(f'DIN 模型参数: {total:,}')
print(model)

---
## Cell 5 — 训练循环（含 Early Stopping）

In [ ]:
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5, verbose=True)

def run_epoch(model, loader, optimizer, criterion, device, training=True):
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            u_feat, u_emb, i_feat, i_emb, seq, mask, label = [
                b.to(device) for b in batch]

            pred = model(u_feat, u_emb, i_feat, i_emb, seq, mask)
            loss = criterion(pred, label)

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss  += loss.item()
            all_preds.extend(pred.detach().cpu().numpy())
            all_labels.extend(label.cpu().numpy())

    avg_loss = total_loss / len(loader)
    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.5
    return avg_loss, auc, all_preds, all_labels


# ── 主训练循环 ──
history = {'train_loss':[], 'val_loss':[], 'train_auc':[], 'val_auc':[]}
best_auc   = 0.0
no_improve = 0
CKPT_PATH  = f"{PATHS['rank_models']}/din_best.pt"

log.info(f'开始训练  epochs={CFG["epochs"]}  device={CFG["device"]}')
print()

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()
    tr_loss, tr_auc, _, _ = run_epoch(
        model, train_loader, optimizer, criterion, CFG['device'], training=True)
    va_loss, va_auc, _, _ = run_epoch(
        model, test_loader,  optimizer, criterion, CFG['device'], training=False)

    scheduler.step(va_auc)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_auc'].append(tr_auc)
    history['val_auc'].append(va_auc)

    flag = '⬆ best' if va_auc > best_auc else ''
    print(f'Epoch {epoch:3d}/{CFG["epochs"]}  '
          f'loss={tr_loss:.4f}/{va_loss:.4f}  '
          f'AUC={tr_auc:.4f}/{va_auc:.4f}  '
          f'{time.time()-t0:.1f}s  {flag}')

    if va_auc > best_auc:
        best_auc = va_auc
        no_improve = 0
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'val_auc': va_auc, 'cfg': CFG}, CKPT_PATH)
    else:
        no_improve += 1
        if no_improve >= CFG['early_stop']:
            log.info(f'Early stopping at epoch {epoch}')
            break

ckpt = torch.load(CKPT_PATH, map_location=CFG['device'])
model.load_state_dict(ckpt['model'])
log.info(f'最佳模型  epoch={ckpt["epoch"]}  val_AUC={ckpt["val_auc"]:.4f}')

---
## Cell 6 — 训练过程可视化

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0d1117')

def styled(ax, title):
    ax.set_facecolor('#161b22')
    ax.set_title(title, color='#c9d1d9', fontsize=11)
    ax.tick_params(colors='#6e7681')
    for s in ax.spines.values(): s.set_color('#30363d')

epochs_ran = range(1, len(history['train_loss']) + 1)

styled(axes[0], 'Loss (BCE)')
axes[0].plot(epochs_ran, history['train_loss'], '#00d4ff', lw=2, label='Train')
axes[0].plot(epochs_ran, history['val_loss'],   '#39d353', lw=2, label='Val', linestyle='--')
axes[0].set_xlabel('Epoch', color='#6e7681')
axes[0].legend()

styled(axes[1], 'AUC')
axes[1].plot(epochs_ran, history['train_auc'], '#00d4ff', lw=2, label='Train')
axes[1].plot(epochs_ran, history['val_auc'],   '#39d353', lw=2, label='Val', linestyle='--')
axes[1].axhline(y=0.5, color='#f0c040', linestyle=':', lw=1, label='Random')
axes[1].set_xlabel('Epoch', color='#6e7681')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{PATHS['reports']}/step3_training_curve.png",
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## Cell 7 — 离线评估（AUC / GAUC / LogLoss）

In [ ]:
# ============================================================
# 精排评估指标
#   AUC   → 整体排序能力
#   GAUC  → 分用户 AUC 平均（更公平，工业标准）
#   LogLoss → 预测概率校准度
#   NDCG@K → 排序质量
# ============================================================
from sklearn.metrics import roc_auc_score, log_loss
from tabulate import tabulate

def compute_gauc(df, preds, labels):
    """GAUC: 分用户计算AUC后按曝光数加权平均"""
    df = df.copy()
    df['pred']  = preds
    df['label'] = labels
    gauc_sum, weight_sum = 0.0, 0
    for uid, group in df.groupby('user_id'):
        if group['label'].nunique() < 2: continue
        auc = roc_auc_score(group['label'], group['pred'])
        w   = len(group)
        gauc_sum  += auc * w
        weight_sum += w
    return gauc_sum / weight_sum if weight_sum > 0 else 0

def compute_ndcg(df, preds, k=10):
    """NDCG@K: 分用户计算后平均"""
    df = df.copy()
    df['pred']  = preds
    ndcg_list = []
    for uid, group in df.groupby('user_id'):
        if group['label'].sum() == 0: continue
        group = group.sort_values('pred', ascending=False).head(k)
        dcg  = sum(row['label'] / np.log2(i+2)
                   for i, (_, row) in enumerate(group.iterrows()))
        idcg = sum(1 / np.log2(i+2)
                   for i in range(min(int(group['label'].sum()), k)))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(ndcg_list) if ndcg_list else 0

# ── 获取测试集预测结果 ──
with Timer('计算评估指标'):
    _, _, all_preds, all_labels = run_epoch(
        model, test_loader, optimizer, criterion,
        CFG['device'], training=False)

    auc     = roc_auc_score(all_labels, all_preds)
    logloss = log_loss(all_labels, all_preds)
    gauc    = compute_gauc(test_df.reset_index(drop=True), all_preds, all_labels)
    ndcg10  = compute_ndcg(test_df.reset_index(drop=True), all_preds, k=10)
    ndcg20  = compute_ndcg(test_df.reset_index(drop=True), all_preds, k=20)

print('\n【DIN 精排模型离线评估】')
rows = [
    ['AUC',      f'{auc:.4f}',     '>0.72 达标'],
    ['GAUC',     f'{gauc:.4f}',    '>0.68 达标'],
    ['LogLoss',  f'{logloss:.4f}', '<0.50 达标'],
    ['NDCG@10',  f'{ndcg10:.4f}',  '越高越好'],
    ['NDCG@20',  f'{ndcg20:.4f}',  '越高越好'],
]
print(tabulate(rows, headers=['指标', '数值', '参考标准'], tablefmt='grid'))

---
## Cell 8 — 完整推荐链路测试（召回 → 精排 → Top10）

In [ ]:
# ============================================================
# 完整推荐链路测试
# 对应企业: 线上推理服务的完整流程
#
# 流程:
#   输入用户ID
#   → 双塔 FAISS 召回 TopK 候选
#   → DIN 精排打分
#   → 返回 Top10 推荐结果
# ============================================================
import faiss

# 加载 Step 2 的 FAISS 索引
faiss_index = faiss.read_index(f"{PATHS['recall_models']}/faiss_item.index")
recall_item_ids = np.load(f"{PATHS['recall_models']}/item_ids.npy")
recall_user_ids = np.load(f"{PATHS['recall_models']}/user_ids.npy")
user_id2row = {uid: i for i, uid in enumerate(recall_user_ids)}

user_emb_matrix = np.stack([
    user_emb_dict.get(uid, np.zeros(EMB_DIM, np.float32))
    for uid in recall_user_ids
]).astype(np.float32)


def recommend(user_id, recall_k=200, top_n=10):
    """
    完整推荐链路
    user_id  → 双塔召回 recall_k 个候选 → DIN精排 → Top top_n
    """
    model.eval()
    device = CFG['device']

    # ── Step1: 双塔召回 ──
    if user_id not in user_id2row:
        return []
    u_vec = user_emb_matrix[user_id2row[user_id]:user_id2row[user_id]+1]
    _, indices = faiss_index.search(u_vec, recall_k)
    candidate_ids = [int(recall_item_ids[i]) for i in indices[0]]

    # ── Step2: DIN 精排 ──
    u_feat_v = torch.tensor(
        user_feat_dict.get(user_id, np.zeros(U_DIM, np.float32)),
        dtype=torch.float).unsqueeze(0).to(device)
    u_emb_v = torch.tensor(
        user_emb_dict.get(user_id, np.zeros(EMB_DIM, np.float32)),
        dtype=torch.float).unsqueeze(0).to(device)

    seq = seq_dict.get(user_id, [])
    seq = seq[-CFG['seq_len']:]
    pad_len = CFG['seq_len'] - len(seq)
    seq_pad  = torch.tensor([0]*pad_len + seq, dtype=torch.long).unsqueeze(0).to(device)
    seq_mask = torch.tensor([0]*pad_len + [1]*len(seq), dtype=torch.float).unsqueeze(0).to(device)

    scores = []
    batch_size = 64
    for start in range(0, len(candidate_ids), batch_size):
        batch_iids = candidate_ids[start:start+batch_size]
        B = len(batch_iids)

        i_feat_b = torch.tensor(
            np.stack([item_feat_dict.get(i, np.zeros(I_DIM, np.float32)) for i in batch_iids]),
            dtype=torch.float).to(device)
        i_emb_b = torch.tensor(
            np.stack([item_emb_dict.get(i, np.zeros(EMB_DIM, np.float32)) for i in batch_iids]),
            dtype=torch.float).to(device)

        u_feat_exp = u_feat_v.expand(B, -1)
        u_emb_exp  = u_emb_v.expand(B, -1)
        seq_exp    = seq_pad.expand(B, -1)
        mask_exp   = seq_mask.expand(B, -1)

        with torch.no_grad():
            pred = model(u_feat_exp, u_emb_exp, i_feat_b, i_emb_b, seq_exp, mask_exp)
        scores.extend(zip(batch_iids, pred.cpu().numpy()))

    # ── Step3: 排序取 Top N ──
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_n]


# ── 测试 3 个用户 ──
test_users = list(user2idx.keys())[:3]
print('\n【完整推荐链路测试】')
print(f'  召回候选数: 200  精排输出: Top 10\n')

for uid in test_users:
    results = recommend(uid, recall_k=200, top_n=10)
    print(f'  用户 {uid} 的推荐结果:')
    for rank, (iid, score) in enumerate(results, 1):
        print(f'    #{rank:2d}  item_id={iid:5d}  score={score:.4f}')
    print()

---
## Cell 9 — 模型保存

In [ ]:
with Timer('保存模型和配置'):
    # 保存完整模型信息
    torch.save({
        'epoch'    : ckpt['epoch'],
        'model'    : model.state_dict(),
        'val_auc'  : ckpt['val_auc'],
        'cfg'      : CFG,
        'u_dim'    : U_DIM,
        'i_dim'    : I_DIM,
        'emb_dim'  : EMB_DIM,
        'n_items'  : N_ITEMS,
    }, f"{PATHS['rank_models']}/din_best.pt")

    # 保存特征 scaler（serving 时需要）
    with open(f"{PATHS['rank_models']}/scalers.pkl", 'wb') as f:
        pickle.dump({'user': user_scaler, 'item': item_scaler}, f)

log.info('模型和配置保存完成')

---
## Cell 10 — Step 3 完成报告

In [ ]:
print('\n' + '='*65)
print('  📋  STEP 3 PIPELINE 完成报告')
print('='*65)

summary = [
    ['模型',         'DIN (Deep Interest Network)', ''],
    ['训练样本数',   f'{len(train_ds):,}',           '条'],
    ['最佳 Epoch',   str(ckpt['epoch']),             ''],
    ['AUC',          f'{auc:.4f}',                   ''],
    ['GAUC',         f'{gauc:.4f}',                  ''],
    ['LogLoss',      f'{logloss:.4f}',               ''],
    ['NDCG@10',      f'{ndcg10:.4f}',               ''],
    ['NDCG@20',      f'{ndcg20:.4f}',               ''],
]
print(tabulate(summary, headers=['指标', '数值', '单位'], tablefmt='simple'))

print('\n【输出文件】')
output_files = [
    f"{PATHS['rank_models']}/din_best.pt",
    f"{PATHS['rank_models']}/scalers.pkl",
    f"{PATHS['reports']}/step3_training_curve.png",
]
for fp in output_files:
    if os.path.exists(fp):
        mb = os.path.getsize(fp) / 1024**2
        print(f'  ✅ {fp.replace(BASE_DIR+"/",""):50s} {mb:.2f} MB')
    else:
        print(f'  ❌ {fp}')

print('\n' + '='*65)
print('  ✅  Step 3 完成！')
print('  ▶   下一步: Step 4 — A/B 实验平台 + 在线监控')
print('='*65)